# Exploratory Data Analysis: MovieLens Latest Small

Before building any recommender, I want to understand the data deeply.
The choices I make later (train/test split strategy, minimum rating thresholds,
which metrics to prioritise) all come from what I find here.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA = Path('../data/raw')

## 1. Load the data

In [ ]:
ratings = pd.read_csv(DATA / 'ratings.csv')
movies  = pd.read_csv(DATA / 'movies.csv')
tags    = pd.read_csv(DATA / 'tags.csv')

print(f'Ratings shape : {ratings.shape}')
print(f'Movies shape  : {movies.shape}')
print(f'Tags shape    : {tags.shape}')
ratings.head()

## 2. Basic statistics

In [ ]:
n_users   = ratings['userId'].nunique()
n_items   = ratings['movieId'].nunique()
n_ratings = len(ratings)
sparsity  = 1 - n_ratings / (n_users * n_items)

print(f'Users    : {n_users:,}')
print(f'Movies   : {n_items:,}')
print(f'Ratings  : {n_ratings:,}')
print(f'Sparsity : {sparsity:.4%}')
print()
print('Rating scale:', ratings['rating'].min(), '→', ratings['rating'].max(),
      '(half-star increments)')

The **98.3% sparsity** is the core challenge for collaborative filtering.
Most user-item pairs are unobserved, the matrix is almost entirely empty.

## 3. Rating distribution

Understanding the rating distribution matters for the matrix factorisation model:
if ratings are heavily skewed, the global bias term needs to account for that.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count per rating value
dist = ratings['rating'].value_counts().sort_index()
axes[0].bar(dist.index, dist.values, width=0.4, color='#e94560')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating distribution')

# Cumulative
axes[1].plot(dist.index, dist.cumsum() / dist.sum(), marker='o', color='#4361ee')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Cumulative %')
axes[1].set_title('Cumulative distribution')

plt.tight_layout()
plt.savefig('../results/figures/rating_distribution.png', bbox_inches='tight')
plt.show()

print(f'Mean rating : {ratings["rating"].mean():.2f}')
print(f'Median      : {ratings["rating"].median()}')
print(f'Std dev     : {ratings["rating"].std():.2f}')

Ratings skew positive, most are 3.0 or above. This is typical of MovieLens:
users tend to rate movies they chose to watch, which introduces a selection bias.
The 4.0 peak suggests users rate good films more than bad ones.

## 4. User activity distribution

How many movies does each user rate? This matters because:
- Users with very few ratings are hard to personalise for (cold-start problem)
- A handful of power users could dominate and bias the model

In [ ]:
user_counts = ratings.groupby('userId')['rating'].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(user_counts, bins=40, color='#4361ee', edgecolor='none')
axes[0].set_xlabel('Ratings per user')
axes[0].set_ylabel('Number of users')
axes[0].set_title('User activity distribution')
axes[0].axvline(user_counts.median(), color='#e94560', linestyle='--',
                label=f'Median = {user_counts.median():.0f}')
axes[0].legend()

# Top 20 most active users
top_users = user_counts.head(20)
axes[1].barh(top_users.index.astype(str), top_users.values, color='#e94560')
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('User ID')
axes[1].set_title('Top 20 most active users')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../results/figures/user_activity.png', bbox_inches='tight')
plt.show()

print(f'Median ratings per user : {user_counts.median():.0f}')
print(f'Max ratings by one user : {user_counts.max():,} (User {user_counts.idxmax()})')
print(f'Users with < 20 ratings : {(user_counts < 20).sum()}')

## 5. Item popularity: the long tail

This is one of the most important plots for understanding recommender systems.
The "long tail" refers to the fact that a small number of blockbuster films
account for most of the ratings, while the vast majority of movies have very few.

In [ ]:
item_counts = ratings.groupby('movieId')['rating'].count().sort_values(ascending=False).reset_index()
item_counts.columns = ['movieId', 'n_ratings']
item_counts['rank'] = range(1, len(item_counts) + 1)
item_counts['cumulative_pct'] = item_counts['n_ratings'].cumsum() / item_counts['n_ratings'].sum()

# What fraction of movies account for 50% of all ratings?
top_50pct = (item_counts['cumulative_pct'] <= 0.5).sum()
pct_movies = top_50pct / len(item_counts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Long tail
axes[0].fill_between(item_counts['rank'], item_counts['n_ratings'],
                     alpha=0.6, color='#4361ee')
axes[0].axvline(top_50pct, color='#e94560', linestyle='--',
                label=f'Top {pct_movies:.1%} of movies\n= 50% of all ratings')
axes[0].set_xlabel('Movie rank (by popularity)')
axes[0].set_ylabel('Number of ratings')
axes[0].set_title('Item popularity , the long tail')
axes[0].legend()

# Cumulative coverage
axes[1].plot(item_counts['rank'] / len(item_counts) * 100,
             item_counts['cumulative_pct'] * 100,
             color='#06d6a0', linewidth=2)
axes[1].set_xlabel('% of catalog (ranked by popularity)')
axes[1].set_ylabel('% of total ratings covered')
axes[1].set_title('Cumulative rating coverage')
axes[1].axhline(50, color='#e94560', linestyle='--', alpha=0.6)
axes[1].axhline(80, color='#f0a500', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('../results/figures/long_tail.png', bbox_inches='tight')
plt.show()

print(f'Top {pct_movies:.1%} of movies account for 50% of all ratings')
top_80 = (item_counts['cumulative_pct'] <= 0.8).sum() / len(item_counts)
print(f'Top {top_80:.1%} of movies account for 80% of all ratings')

## 6. Genre analysis

In [ ]:
genre_series = movies['genres'].str.split('|').explode()
genre_counts = genre_series[genre_series != '(no genres listed)'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count of movies per genre
genre_counts.plot(kind='barh', ax=axes[0], color='#4361ee')
axes[0].set_xlabel('Number of movies')
axes[0].set_title('Movies per genre')
axes[0].invert_yaxis()

# Average rating per genre
movies_ratings = ratings.merge(movies, on='movieId')
genre_avg = (
    movies_ratings.assign(genre=movies_ratings['genres'].str.split('|'))
    .explode('genre')
    .query('genre != "(no genres listed)"')
    .groupby('genre')['rating']
    .agg(['mean', 'count'])
    .query('count >= 100')
    .sort_values('mean')
)
genre_avg['mean'].plot(kind='barh', ax=axes[1], color='#e94560')
axes[1].set_xlabel('Average rating')
axes[1].set_title('Average rating per genre (min 100 ratings)')
axes[1].axvline(ratings['rating'].mean(), color='white', linestyle='--',
                alpha=0.4, label='Global mean')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/genre_analysis.png', bbox_inches='tight')
plt.show()

## 7. Temporal analysis: ratings over time

In [ ]:
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year'] = ratings['date'].dt.year

yearly = ratings.groupby('year').agg(
    n_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean'),
    n_users=('userId', 'nunique'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(yearly['year'], yearly['n_ratings'], color='#4361ee')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of ratings')
axes[0].set_title('Ratings per year')

axes[1].plot(yearly['year'], yearly['avg_rating'], marker='o', color='#e94560')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Average rating')
axes[1].set_title('Average rating per year')
axes[1].set_ylim(3.0, 4.2)

plt.tight_layout()
plt.savefig('../results/figures/temporal_analysis.png', bbox_inches='tight')
plt.show()

## 8. Train / test split rationale

I use a **temporal split** rather than a random one.
For each user, the most recent 20% of their ratings go to the test set.
This reflects how a real system would be evaluated: you always predict future
behaviour from past behaviour, never the other way around.

A random split would leak future ratings into training, making results
look better than they actually are.

In [ ]:
ratings_sorted = ratings.sort_values(['userId', 'timestamp'])

def split_user(group):
    n       = len(group)
    cutoff  = max(1, int(n * 0.8))
    splits  = ['train'] * cutoff + ['test'] * (n - cutoff)
    group   = group.reset_index(drop=True)
    group['split'] = splits
    return group

split_df = ratings_sorted.groupby('userId', group_keys=False).apply(
    split_user, include_groups=False
)
split_df['userId'] = ratings_sorted['userId'].values

train = split_df[split_df['split'] == 'train']
test  = split_df[split_df['split'] == 'test']

print(f'Train : {len(train):,} ratings ({len(train)/len(ratings):.1%})')
print(f'Test  : {len(test):,}  ratings ({len(test)/len(ratings):.1%})')

# Show the cutpoint for a sample user
sample_user = split_df.groupby('userId').size().nlargest(1).index[0]
user_df = split_df[split_df['userId'] == sample_user].copy()

fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(user_df[user_df['split']=='train']['date'],
           user_df[user_df['split']=='train']['rating'],
           c='#4361ee', alpha=0.5, s=20, label='Train')
ax.scatter(user_df[user_df['split']=='test']['date'],
           user_df[user_df['split']=='test']['rating'],
           c='#e94560', alpha=0.8, s=20, label='Test')
ax.set_title(f'Temporal split for User {sample_user} (most active user)')
ax.set_xlabel('Date')
ax.set_ylabel('Rating')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/train_test_split.png', bbox_inches='tight')
plt.show()

## Summary

| Finding | Implication for modelling |
|---|---|
| 98.3% sparsity | CF will struggle; matrix factorisation handles this better |
| Ratings skew positive (mean 3.5) | Need mean-centring in content-based user profiles |
| Long tail: top 3% = 50% of ratings | Most Popular baseline is strong; coverage metric is essential |
| Heavy power users (up to 2,698 ratings) | Could dominate similarity; worth monitoring |
| Film-Noir / War rate highest on average | Genre matters for quality: content-based has signal to exploit |